# NB3 — Time Travel + MERGE Upsert (lightweight)

Maps to slide §3 + deliverable bullet 3.

> Spark equivalent: `option("versionAsOf", N)` ↔ `DeltaTable(path, version=N)`
>                   `MERGE INTO ...`           ↔ `dt.merge(source, predicate)`

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
import time
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "customers_tt")
reset(table_path)

## 1. Build version history
v0: initial 100K · v1: schema add · v2: MERGE upsert · v3: bad data

In [2]:
# v0 — initial load
v0 = pl.DataFrame({
    "customer_id": list(range(100_000)),
    "status":      ["active"] * 100_000,
    "score":       [i % 1000 for i in range(100_000)],
})
write_deltalake(table_path, v0.to_arrow(), mode="overwrite")

# v1 — add `tier` column (schema evolution)
v1 = (pl.from_arrow(DeltaTable(table_path).to_pyarrow_table())
        .with_columns(
            pl.when(pl.col("score") > 800).then(pl.lit("gold")).otherwise(pl.lit("silver")).alias("tier")
        ))
write_deltalake(table_path, v1.to_arrow(), mode="overwrite", schema_mode="overwrite")

# v2 — MERGE upsert 100K (50K updates, 50K inserts)
updates = pl.DataFrame({
    "customer_id": list(range(50_000, 150_000)),
    "status":      ["vip"] * 100_000,
    "score":       [999] * 100_000,
    "tier":        ["platinum"] * 100_000,
})
t0 = time.time()
(DeltaTable(table_path)
    .merge(source=updates.to_arrow(),
           predicate="t.customer_id = s.customer_id",
           source_alias="s", target_alias="t")
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute())
print(f"MERGE 100K rows: {time.time()-t0:.2f}s")

# v3 — simulate bad data
bad = pl.DataFrame({
    "customer_id": list(range(50)),
    "status":      [None] * 50,
    "score":       [-1] * 50,
    "tier":        ["UNKNOWN"] * 50,
}, schema={"customer_id": pl.Int64, "status": pl.Utf8, "score": pl.Int64, "tier": pl.Utf8})
write_deltalake(table_path, bad.to_arrow(), mode="append")

MERGE 100K rows: 0.15s


### 📝 Nhận xét — Xây dựng lịch sử version

Cell này tạo 4 commit, mỗi commit là một version:

| Version | Thao tác | Ý nghĩa |
|---|---|---|
| v0 | `WRITE overwrite` | Nạp 100.000 khách hàng, tất cả `active` |
| v1 | `WRITE overwrite` + `schema_mode="overwrite"` | Thêm cột `tier` bằng cách **ghi lại toàn bộ bảng** (khác NB1: `schema_mode="merge"` chỉ mở rộng schema, không rewrite file cũ) |
| v2 | **`MERGE`** | Upsert 100.000 dòng nguồn (`customer_id` 50.000–149.999) |
| v3 | `WRITE append` | Mô phỏng sự cố: 50 dòng lỗi `score=-1`, `status=null` |

**MERGE (upsert)** ghép nguồn `s` với bảng đích `t` theo `t.customer_id = s.customer_id`:
- id 50.000–99.999 **khớp** → `when_matched_update_all()` → **update** thành `vip`/`platinum`.
- id 100.000–149.999 **không khớp** → `when_not_matched_insert_all()` → **insert** dòng mới.
- Hoàn thành 100K dòng trong **0,15 s** (lightweight path, 1 file đích).

MERGE là cách ghi **idempotent**: chạy lại cùng lô dữ liệu vẫn cho cùng kết quả, không sinh dòng trùng — khác với `append` (NB1: chạy lại append làm `dan` bị trùng; ở đây v3 append cũng tạo 50 id trùng).


## 2. history() — audit trail

In [3]:
for h in DeltaTable(table_path).history():
    print(f"  v{h['version']:>2}  {h['operation']:<25}  metrics={h.get('operationMetrics', {})}")

  v 3  WRITE                      metrics={'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 50, 'execution_time_ms': 3, 'num_retries': 0}
  v 2  MERGE                      metrics={'num_source_rows': 100000, 'num_target_rows_inserted': 50000, 'num_target_rows_updated': 50000, 'num_target_rows_deleted': 0, 'num_target_rows_copied': 50000, 'num_output_rows': 150000, 'num_target_files_scanned': 1, 'num_target_files_skipped_during_scan': 0, 'num_target_files_added': 1, 'num_target_files_removed': 1, 'execution_time_ms': 127, 'scan_time_ms': 85, 'rewrite_time_ms': 22}
  v 1  WRITE                      metrics={'num_added_files': 1, 'num_removed_files': 1, 'num_partitions': 0, 'num_added_rows': 100000, 'execution_time_ms': 27, 'num_retries': 0}
  v 0  WRITE                      metrics={'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 100000, 'execution_time_ms': 41, 'num_retries': 0}


### 📝 Nhận xét — history() là audit trail

`history()` đọc `commitInfo` của từng commit trong `_delta_log/`, từ mới đến cũ. Metrics của **MERGE (v2)** cho biết chính xác chuyện gì đã xảy ra:
- `num_source_rows: 100000` → `num_target_rows_updated: 50000` + `num_target_rows_inserted: 50000` ⇒ `num_output_rows: 150000`.
- `num_target_rows_copied: 50000` và `num_target_files_removed: 1`, `num_target_files_added: 1`: Parquet là file **bất biến**, không sửa từng dòng tại chỗ được. MERGE đọc file cũ, **chép lại 50.000 dòng không đổi** (id 0–49.999) cùng các dòng update/insert vào **file mới**, rồi `remove` file cũ — cơ chế **copy-on-write**. Chi phí MERGE tỉ lệ với kích thước file bị chạm, không chỉ số dòng thay đổi.
- `num_target_files_skipped_during_scan: 0`: bảng chỉ có 1 file nên không có gì để bỏ qua; với bảng lớn đã Z-order (NB2), MERGE có thể dùng stats để chỉ rewrite các file liên quan.

v3 (`WRITE`, `num_added_rows: 50`) là commit chứa dữ liệu lỗi — history cho biết **khi nào, thao tác gì** đã đưa nó vào.


## 3. Time-travel queries

In [4]:
v0_count = DeltaTable(table_path, version=0).to_pyarrow_table().num_rows
v1_cols  = DeltaTable(table_path, version=1).schema().to_arrow().names
print(f"v0 row count: {v0_count}")
print(f"v1 schema:    {v1_cols}")

v0 row count: 100000
v1 schema:    ['customer_id', 'status', 'score', 'tier']


### 📝 Nhận xét — Time travel

- `DeltaTable(path, version=0)` trả **100.000 dòng** dù bảng đã bị overwrite (v1), merge (v2) và append (v3); `version=1` cho schema có `tier` (v0 chưa có).
- **Cơ chế:** reader replay log **chỉ đến commit N** để biết tập file thuộc version đó, rồi đọc đúng các file ấy. Các file của version cũ **vẫn nằm trên đĩa** — chúng chỉ bị đánh dấu `remove` trong log ở commit sau, không bị xoá vật lý.
- Ứng dụng: tái lập báo cáo/ML training trên đúng snapshot dữ liệu cũ, so sánh trước/sau một thay đổi, điều tra sự cố.


## 4. RESTORE bad version (rollback)

`restore(2)` rewinds the *current* state of the table to whatever it was
at version 2, recorded as a new version (v4). The old versions stay in
history — restore is itself a transaction, fully auditable.

In [5]:
t0 = time.time()
dt = DeltaTable(table_path)
dt.restore(2)
print(f"RESTORE → v2: {time.time()-t0:.2f}s   (target < 30s)")

# Verify the bad rows are gone — use delta-rs's native filter pushdown.
# (DuckDB's delta extension as of 1.5.x is stricter about post-RESTORE
# protocol entries than delta-rs writes; routing through delta-rs end-to-end
# avoids the InvalidProtocolError race.)
dt_after = DeltaTable(table_path)
bad_count = dt_after.to_pyarrow_table(filters=[("score", "<", 0)]).num_rows
print(f"Rows with score<0 after restore: {bad_count}  (expected 0)")

RESTORE → v2: 0.02s   (target < 30s)
Rows with score<0 after restore: 0  (expected 0)


### 📝 Nhận xét — RESTORE (rollback)

- Trước restore, v3 có **50 dòng `score < 0`** (và 50 `customer_id` trùng, `status = null`). Sau `restore(2)`: **0 dòng `score < 0`** ✅.
- Schema enforcement **không chặn** được lỗi này: `-1` vẫn là Int64 hợp lệ, `null` hợp lệ với cột nullable. Enforcement kiểm tra **kiểu**, không kiểm tra **nghiệp vụ** — vì vậy cần khả năng rollback.
- RESTORE chỉ mất **0,02 s** vì nó **không chép dữ liệu**: v2 và v3 dùng chung file của MERGE, v3 chỉ thêm 1 file lỗi. Để quay về trạng thái v2, Delta chỉ cần ghi một commit `remove` file lỗi đó (xem nội dung `…004.json` ở cell dòng thời gian bên dưới).
- `restore(2)` **không xoá v3** — nó tạo **version mới v4** có nội dung giống v2.


## 5. history() — final audit trail (now includes the RESTORE)

In [6]:
final_history = DeltaTable(table_path).history()
for h in final_history:
    print(f"  v{h['version']:>2}  {h['operation']:<25}")
print(f"\nTotal versions: {len(final_history)}  (target ≥ 5)")

  v 4  RESTORE                  
  v 3  WRITE                    
  v 2  MERGE                    
  v 1  WRITE                    
  v 0  WRITE                    

Total versions: 5  (target ≥ 5)


### 📝 Nhận xét — History sau RESTORE

- **5 version** (v0–v4), dòng trên cùng là `v4 RESTORE` ⇒ đạt "≥ 5 version, gồm RESTORE" ✅.
- RESTORE là một **transaction mới**, không viết lại lịch sử: v3 với dữ liệu lỗi vẫn còn trong history và vẫn time travel về được để điều tra (ai/khi nào ghi 50 dòng lỗi). Mọi rollback đều được audit.


In [7]:
# Dòng thời gian: đọc lại từng version bằng time travel
import glob
import json
from pathlib import Path

for v in range(DeltaTable(table_path).version() + 1):
    dtv = DeltaTable(table_path, version=v)
    df = pl.from_arrow(dtv.to_pyarrow_table())
    print(f"v{v}  rows={df.height:>6}  files={len(dtv.file_uris())}  "
          f"score<0={df.filter(pl.col('score') < 0).height:>3}  "
          f"dup_ids={df.height - df['customer_id'].n_unique():>3}  "
          f"status={dict(sorted(df.group_by('status').len().iter_rows(), key=str))}")

# Commit RESTORE thực sự ghi gì vào log?
last_log = sorted(glob.glob(f"{table_path}/_delta_log/*.json"))[-1]
print(f"\nNội dung {Path(last_log).name}:")
for line in open(last_log):
    action = json.loads(line)
    kind = next(iter(action))
    if kind == "commitInfo":
        ci = action[kind]
        print(f"  commitInfo  operation={ci['operation']} params={ci['operationParameters']} "
              f"readVersion={ci.get('readVersion')} metrics={ci.get('operationMetrics')}")
    elif kind in ("add", "remove"):
        print(f"  {kind:<11} {action[kind]['path'][:45]}…  size={action[kind].get('size')}")
    else:
        print(f"  {kind}")

print(f"\nFile Parquet trên đĩa: {len(glob.glob(f'{table_path}/*.parquet'))}  |  "
      f"file version hiện tại dùng: {len(DeltaTable(table_path).file_uris())}")

v0  rows=100000  files=1  score<0=  0  dup_ids=  0  status={'active': 100000}


v1  rows=100000  files=1  score<0=  0  dup_ids=  0  status={'active': 100000}


v2  rows=150000  files=1  score<0=  0  dup_ids=  0  status={'active': 50000, 'vip': 100000}


v3  rows=150050  files=2  score<0= 50  dup_ids= 50  status={'active': 50000, 'vip': 100000, None: 50}


v4  rows=150000  files=1  score<0=  0  dup_ids=  0  status={'active': 50000, 'vip': 100000}

Nội dung 00000000000000000004.json:
  commitInfo  operation=RESTORE params={'version': '2'} readVersion=3 metrics={'numRemovedFile': 1, 'numRestoredFile': 0}
  protocol
  remove      part-00000-beb4a824-7db7-49ed-a5e4-5c99c2046c…  size=1570
  metaData

File Parquet trên đĩa: 4  |  file version hiện tại dùng: 1


### 📝 Nhận xét — Dòng thời gian và commit RESTORE

| Version | Rows | Files | `score<0` | id trùng | Status |
|---|---:|---:|---:|---:|---|
| v0 | 100.000 | 1 | 0 | 0 | 100K active |
| v1 | 100.000 | 1 | 0 | 0 | 100K active (+ cột `tier`) |
| v2 | 150.000 | 1 | 0 | 0 | 50K active + 100K vip |
| v3 | 150.050 | **2** | **50** | **50** | + 50 `null` |
| v4 | 150.000 | 1 | 0 | 0 | giống hệt v2 |

- Commit `…004.json` chỉ chứa: `commitInfo` (`operation=RESTORE`, `version: '2'`, `readVersion=3`, `numRemovedFile: 1`, `numRestoredFile: 0`), `protocol`, **một action `remove`** cho file 1.570 bytes chứa 50 dòng lỗi, và `metaData`. Không có `add` nào ⇒ không rewrite dữ liệu.
- **4 file Parquet trên đĩa** (mỗi version v0–v3 ghi 1 file) nhưng version hiện tại chỉ dùng **1**. 3 file còn lại là thứ giữ cho time travel hoạt động. Nếu chạy `VACUUM` với retention ngắn, chúng bị xoá vật lý và **không thể time travel về v0, v1, v3 nữa** — đánh đổi giữa chi phí lưu trữ và khả năng khôi phục (NB6).
- Lưu ý: RESTORE đưa **toàn bộ bảng** về trạng thái cũ, khác `git revert` (chỉ đảo một thay đổi). Nếu sau v3 có thêm dữ liệu tốt, restore về v2 cũng sẽ làm mất dữ liệu tốt đó — production cần cân nhắc dùng `DELETE`/`MERGE` có điều kiện thay vì restore.


## ✅ Deliverable check
- [ ] history() shows ≥ 5 versions (incl. RESTORE itself)
- [ ] MERGE 100K finished in < 60s (likely < 1s on lightweight path)
- [ ] RESTORE finished in < 30s and removed bad rows

In [8]:
ops = [h["operation"] for h in final_history]
checks = {
    "history ≥ 5 versions":          len(final_history) >= 5,
    "history includes the RESTORE":  any("RESTORE" in o.upper() for o in ops),
    "MERGE recorded in history":     any("MERGE" in o.upper() for o in ops),
    "bad rows gone after restore":   bad_count == 0,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB3 incomplete — see FAIL rows above"
print("\nNB3 complete.")

  [PASS] history ≥ 5 versions
  [PASS] history includes the RESTORE
  [PASS] MERGE recorded in history
  [PASS] bad rows gone after restore

NB3 complete.


### 🎯 Kết luận NB3

Transaction log biến bảng thành một chuỗi version bất biến. **MERGE** upsert 100K dòng (50K update + 50K insert) trong một commit nguyên tử bằng copy-on-write và là cách ghi idempotent. **Time travel** đọc lại bất kỳ version nào vì file cũ chưa bị xoá vật lý. Khi dữ liệu lỗi lọt qua schema enforcement (kiểu hợp lệ, nghiệp vụ sai), **RESTORE** quay về v2 bằng một commit mới chỉ `remove` 1 file — nhanh, không mất lịch sử, có audit trail. Khả năng này tồn tại chừng nào file cũ còn trên đĩa — retention của VACUUM (NB6) quyết định cửa sổ khôi phục.
